In [ ]:
####################################
########## Brayan Mora A. ########## 
##########  5/04/2025     ##########
####################################

#  Este codigo realiza el control de calidad de los datos teniendo en cuenta lo siguiente 
#  1. Tiene en cuenta que los datos pueden estar en un folder o en un subfolder, lo que hace es buscar un archivo dentro 
#     De las multiples estructuras que puedan tener los folders, esta busqueda se da teniendo en cuenta que los archivos puedam 
#     ser, tif , tiff, nc 
#  2. Valida que los datos no esten corruptos, es decir que se puedan abrir y operar
#  3. Valida que por lo menos haya un valor dentro del raster que sea diferente de NA 
#  5. Guarda los archivos en un temporal el cual se guardan teniendo en cuenta el periodo de analisis.
#  6. Genera un log que se encarga de reportar si hay errores o todo se realizo con exito 
# 

import os
import shutil
import rasterio
from rasterio.errors import RasterioIOError
import numpy as np
from datetime import datetime
import re

def quality_control(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    log_path = os.path.join(output_dir, 'log_revision_rasters.txt')

    def encontrar_raster_en_directorio(directorio):
        for root, dirs, files in os.walk(directorio):
            for file in files:
                if file.lower().endswith(('.tif', '.tiff', '.nc')):
                    return os.path.join(root, file)
        return None

    def extraer_anos(nombre_folder):
        match = re.search(r'(\d{4})[^\d]*(\d{4})', nombre_folder)
        if match:
            return match.group(1), match.group(2)
        return None, None

    def procesar_raster(raster_path):
        try:
            if raster_path.lower().endswith(('.tif', '.tiff')):
                with rasterio.open(raster_path) as src:
                    array = src.read(1)
                    nodata = src.nodata
                    crs = src.crs
            elif raster_path.lower().endswith('.nc'):
                raise NotImplementedError("El soporte para archivos .nc aún no está implementado.")
            else:
                return False, "Formato de archivo no soportado"

            valores_validos = array[(array != 0) & (array != nodata)]
            if valores_validos.size > 0:
                return True, f"Archivo válido con valores distintos de 0 y nodata. CRS: {crs}"
            else:
                return False, f"Archivo abierto, pero sin valores válidos. CRS: {crs}"
        except RasterioIOError as e:
            return False, f"Error al abrir el archivo: {e}"

    resultados_log = []
    errores = 0
    procesados = 0

    for folder in os.listdir(input_dir):
        folder_path = os.path.join(input_dir, folder)
        if os.path.isdir(folder_path):
            resultados_log.append(f"--- {folder} ---")
            raster_path = encontrar_raster_en_directorio(folder_path)
            if raster_path:
                exito, mensaje = procesar_raster(raster_path)
                resultados_log.append(f"Archivo: {os.path.basename(raster_path)}")
                resultados_log.append(f"Resultado: {mensaje}\n")
                if exito:
                    anio1, anio2 = extraer_anos(folder)
                    if anio1 and anio2:
                        nombre_archivo = f"SMBYC_{anio1}_{anio2}.tif"
                    else:
                        nombre_archivo = f"SMBYC_{folder}.tif"
                    destino = os.path.join(output_dir, nombre_archivo)
                    shutil.copy(raster_path, destino)
                    procesados += 1
                else:
                    errores += 1
            else:
                resultados_log.append("No se encontró ningún archivo válido en el directorio.\n")
                errores += 1

    with open(log_path, 'w') as log_file:
        log_file.write(f"LOG DE REVISIÓN DE RASTERS - {datetime.now()}\n\n")
        log_file.write("\n".join(resultados_log))
        log_file.write("\n\nResumen:\n")
        log_file.write(f"Total procesados correctamente: {procesados}\n")
        log_file.write(f"Total con errores o vacíos: {errores}\n")




    INPUT_DIR = '/home/bmora/ganabosques2/ganabosques_data/1.Data/RAW/20240913/inputs/def/content/'
    OUTPUT_DIR = '/home/bmora/ganabosques_project/Data/def/tmp_content/'
    quality_control(INPUT_DIR, OUTPUT_DIR)


OSError: [Errno 28] No space left on device: '/home/bmora/ganabosques2/ganabosques_data/1.Data/RAW/20240913/inputs/def/content/Cambio_Cobertura_Bosque_No_Bosque_2018_2019/reprojected_2018-2019.tif' -> '/home/bmora/ganabosques_project/Data/def/tmp_content/SMBYC_2018_2019.tif'

In [2]:
df -h

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


NameError: name 'df' is not defined